# 리포트 17 — 도전성 재질이 면적의 45.3% 로 Σ|Γ|A 의 73.5% 를 낸다

> ### 한 일
> **부품 그룹마다 재질을 붙여 7기체의 표면적을 재질로 나누고, 같은 면적을 반사계수 크기로 가중해 무엇이 반사 진폭을 내는지까지 셌다.**

### 결과
1. 재질은 6 가지 [^1] 이고, 7기체 합계 표면적은 3.236 m² [^2] 다.
2. 도전성 carbon, metal, camera_assembly, pcb [^3] 가 면적의 45.3% [^4] 를 차지하고 반사계수 가중 면적 Σ Γ·A 의 73.5% [^5] 를 낸다.
3. 유전체 셸(body·canopy)은 반사계수 크기 0.28 [^6] 로 왕복 투과 τ = 1−Γ², 즉 0.71 dB [^7] (수직입사 기준) 를 곱해 통과시키고 그 뒤 금속(배터리·PCB)을 코히런트 합산한다(`src/rcs_sbr.py` `rcs_sbr(penetrate=True)`).
4. Sionna RT 와 우리 PO 적분기는 **같은 재질 표**를 읽는다(`src/materials.py` `MATERIALS`, `src/drones.py` `DRONE_GROUP_MAT`) — 전파 계산과 산란 계산이 같은 물성에서 나온다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 재질 배정 | 부품 그룹마다 재질을 붙인다 — 그룹은 메쉬를 지을 때 유지한 그 그룹이고, 재질 표는 전파 쪽과 공유한다 |
| 면적 계수 | 7기체의 메쉬 표면적을 재질별로 합산한다 |
| 반사계수 가중 면적 | 같은 면적을 반사계수 크기로 가중한 장부다 — PO 적분에 들어가는 양이고, 위상이 없으므로 σ 자체는 아니다 |
| PO 실효 반사계수 | 수직입사 실효값이다 — 커널은 여기에 각도 모양 \|Γ(θ)\|/\|Γ(0)\| (TE·TM 전력평균, `ANGLE_GAMMA=1` 기본)을 곱한다 [^8] |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python src/viz_mesh_material.py
PYTHONPATH=src python src/build_part03_target_mesh.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json` |
| 소요 | 약 2분 (GPU 0장) |
| 비고 | 재질 표의 정의는 `outputs/report02_derived.json:material.definition` 에 문장으로 적혀 있다 |

---

## 부품별 재질 — PO 적분에 들어가는 물리 입력

Sionna RT 와 우리 PO 적분기는 **같은 재질 표**를 읽는다(`src/materials.py` `MATERIALS`, `src/drones.py` `DRONE_GROUP_MAT`). 아래 그림은 7기체의 표면적을 재질로 나누고, 같은 면적을 |Γ| 로 가중해 **무엇이 반사 진폭을 내는지**까지 함께 싣는다.

## 무엇으로 이루어져 있고, 무엇이 진폭을 내는가

![mesh_compare_material_area](../outputs/figures/mesh_compare_material_area.png)

**그림 1.** 기체는 무엇으로 이루어져 있고, 그 중 무엇이 반사 진폭을 내는가?

도전성 carbon, metal, camera_assembly, pcb [^3] 가 면적의 45.3% [^4] 를 차지하고 Σ|Γ|A 의 73.5% [^5] 를 낸다.

| 재질 | 부품 그룹 | \|Γ\| 벌크 | \|Γ\| PO 실효 | 면적 [m²] (7기체) | 면적 비중 | Σ\|Γ\|A 비중 |
|---|---|---|---|---|---|---|
| carbon | arm, deck, gear_cf | 0.989 | 0.90 | 0.558 | 17.2 % | 27.6 % |
| metal | battery, motor | 1.000 | 1.00 | 0.480 | 14.8 % | 26.3 % |
| plastic | accent, body, canopy, gear | 0.244 | 0.28 | 1.318 | 40.7 % | 20.3 % |
| camera_assembly | camera | 1.000 | 0.85 | 0.311 | 9.6 % | 14.5 % |
| prop_plastic | prop | 0.244 | 0.25 | 0.452 | 14.0 % | 6.2 % |
| pcb | fc, pcb | 1.000 | 0.80 | 0.117 | 3.6 % | 5.1 % |

출처 [^9]

표의 |Γ| 두 열은 **수직입사** 값이다 — 커널은 여기에 각도 모양 |Γ(θ)|/|Γ(0)| (TE·TM 전력평균, `ANGLE_GAMMA=1` 기본)을 곱한다 [^8]. 그 축이 옮기는 크기는 프롭 채널 레벨 +5.16 [^10] ~ +6.48 dB [^11] · 전체 드론 σ +0.08 [^12] ~ +0.10 dB [^13] 다.

## 셸을 통과한 뒤 금속을 더한다

유전체 셸(body·canopy)은 |Γ| = 0.28 [^6] 로 왕복 투과 τ = 1−|Γ|², 즉 0.71 dB [^7] 를 곱해 통과시키고 그 뒤 금속(배터리·PCB)을 코히런트 합산한다 — τ 는 수직입사 기준이고, 각도 모양은 |Γ(θ)| 축이 따로 든다(`src/rcs_sbr.py` `rcs_sbr(penetrate=True)`).

⚠ Σ|Γ|A 는 **위상 없는 장부**다 — 어느 재질이 진폭에 얼마나 기여하는지의 크기 순서를 보는 데 쓰고, σ 자체는 위상을 가진 면적분에서 나온다. 그 적분이 [편 18 «가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다»](18_kernel-what.ipynb) 다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 이 재질 표를 실은 채로 조명면 면적분을 돌린다 | 재질 가중이 σ 에 실제로 얼마나 실리는지가 확정된다 | [편 18 «가림 판정은 Sionna 광선엔진이 하고»](18_kernel-what.ipynb) |
| PO 실효 \|Γ\| 와 벌크 \|Γ\| 의 차이가 σ 에 주는 크기를 잰다 | 실효값 선택이 어느 밴드에서 몇 dB 를 옮기는지가 확정된다 | [편 29 «공통모드 σ 오차는 파형 순위를 안 건드리고»](29_sigma-robustness.ipynb) |
| 셸 투과 모형을 실측 기체 한 대에서 확인한다 | 왕복 투과 가정이 실제 기체에서 성립하는 범위가 확정된다 | [편 74 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](74_sim-vs-meas.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 13개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report02_derived.json` | `material.n_materials` | 6 |
| [^2] | `outputs/report02_derived.json` | `material.total_area_m2` | 3.236 |
| [^3] | `outputs/report02_derived.json` | `material.conducting` | carbon, metal, camera_assembly, pcb |
| [^4] | `outputs/report02_derived.json` | `material.conducting_area_pct` | 45.29 |
| [^5] | `outputs/report02_derived.json` | `material.conducting_gamma_pct` | 73.54 |
| [^6] | `outputs/report02_derived.json` | `material.gamma_shell` | 0.28 |
| [^7] | `outputs/report02_derived.json` | `material.shell_tau_db` | 0.7092 |
| [^8] | `outputs/angle_gamma_impact.json` | `_meta.design` | \|Γ(θ)\| = \|Γ_보정\| · \|Γ_벌크(θ)\|/\|Γ_벌크(0)\| — 수직입사에서 비트 동일 |
| [^9] | `outputs/report02_derived.json` | `material.rows` | (6행 표) |
| [^10] | `outputs/angle_gamma_impact.json` | `propeller_channel_el_-15_3p5GHz.matrice4e.level_delta_db` | 5.16 |
| [^11] | `outputs/angle_gamma_impact.json` | `propeller_channel_el_-15_3p5GHz.mini5pro.level_delta_db` | 6.48 |
| [^12] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.mini5pro.delta_db` | 0.08 |
| [^13] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.matrice4e.delta_db` | 0.1 |